# 06 - Pairwise judging with Kaggle models (Task 7) + report assets (Task 8)
**Create this as a Kaggle *Benchmark* notebook** (kaggle.com/benchmarks → create a task) so `kaggle_benchmarks` and the model proxy are available. **Accelerator:** none needed · **Inputs:** outputs of notebooks 02, 03a, 03b, 04, 05 · **Runtime:** ~30-60 min

This notebook only *judges saved pairwise files*; it never regenerates model outputs.

Judging is resumable: if the session dies, rerun - completed pairs are reused. Save a version when done.

In [ ]:
# ==== EDIT THIS: your GitHub fork of the repository ====
REPO_URL = "https://github.com/Malik9102/small-model-alignment.git"
REPO_DIR = "/kaggle/working/small-model-alignment"

import glob, os, shutil, subprocess, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=False)

def sh(command):
    """Run a shell command, stream its output, and STOP the notebook if it fails.
    (A plain `!command` keeps going after an error, so a crashed training run would go unnoticed.)"""
    print(f"$ {command}", flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}: {command}")

def find_input(pattern):
    """Locate a file/dir inside attached Kaggle inputs (any dataset/notebook slug)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"Nothing matching {pattern!r} under /kaggle/input - did you attach the right notebook output?")
    if len(hits) > 1:
        print(f"[find_input] {len(hits)} matches for {pattern!r}, using {hits[0]}")
    return hits[0]

In [ ]:
sh(f"""python -m pip install -q pyyaml matplotlib""")

In [ ]:
import kaggle_benchmarks as kbench
print('models visible to kaggle_benchmarks:', sorted(kbench.llms))
JUDGE = 'google/gemini-3.6-flash'   # if not listed above, pick a listed model and record it in the report

In [ ]:
# Gather everything from the attached inputs (metrics only - no checkpoints needed here)
for sub in ('generations', 'eval'):
    src = os.path.dirname(find_input(f'outputs/{sub}/' + ('sft_alpaca.json' if sub == 'generations' else 'dpo_uf_vs_sft_pairs.jsonl')))
    shutil.copytree(src, f'outputs/{sub}', dirs_exist_ok=True)
persona_src = os.path.dirname(find_input('outputs/generations/lora_persona.json'))
for name in ('lora_persona.json', 'lora_persona.meta.json', 'smollm_base_persona.json', 'smollm_base_persona.meta.json'):
    shutil.copy(os.path.join(persona_src, name), 'outputs/generations/')
shutil.copy(find_input('outputs/eval/lora_vs_base_persona_pairs.jsonl'), 'outputs/eval/')  # built in notebook 04
for run in ('sft', 'dpo_hh', 'dpo_uf', 'lora_persona'):
    src = os.path.dirname(find_input(f'outputs/{run}/summary.json'))
    os.makedirs(f'outputs/{run}', exist_ok=True)
    for name in ('summary.json', 'config_used.json', 'metrics.jsonl'):
        if os.path.exists(os.path.join(src, name)): shutil.copy(os.path.join(src, name), f'outputs/{run}/')
sh(f"""ls outputs/*""")

## Judge (5 pairs/request, ≥5 s between attempts, 3 attempts per batch, failures logged, resumable)

In [ ]:
from alignment.judging import run_judging
JOBS = [
    ('dpo_uf_vs_sft',            'general'),
    ('dpo_hh_vs_sft',            'safety'),
    ('lora_vs_base_persona',     'persona_style'),
    ('lora_vs_base_persona',     'persona_quality'),
    ('dpo_uf_vs_sft_mirrored',   'general'),   # position-bias check (100 swapped pairs)
    ('dpo_hh_vs_sft_mirrored',   'safety'),
]
for name, rubric in JOBS:
    suffix = '' if rubric in ('general', 'safety') else f'_{rubric}'
    run_judging(
        pairs_path=f'outputs/eval/{name}_pairs.jsonl',
        output_path=f'outputs/eval/{name}{suffix}_judged.jsonl',
        judge_model=JUDGE, batch_size=5, sleep_seconds=5, max_retries=3,
        reuse_judgments=True, rubric=rubric, backend='kbench',
    )

In [ ]:
sh(f"""python scripts/summarize_judgments.py --judged outputs/eval/dpo_uf_vs_sft_judged.jsonl --name 'UF-DPO vs SFT (AlpacaEval 300)' --mirror_judged outputs/eval/dpo_uf_vs_sft_mirrored_judged.jsonl""")
sh(f"""python scripts/summarize_judgments.py --judged outputs/eval/dpo_hh_vs_sft_judged.jsonl --name 'HH-DPO vs SFT (HH eval 300)' --mirror_judged outputs/eval/dpo_hh_vs_sft_mirrored_judged.jsonl""")
sh(f"""python scripts/summarize_judgments.py --judged outputs/eval/lora_vs_base_persona_persona_style_judged.jsonl outputs/eval/lora_vs_base_persona_persona_quality_judged.jsonl --name 'LoRA vs base: style' 'LoRA vs base: quality/safety'""")

## Report assets: plots, tables, examples

In [ ]:
sh(f"""python scripts/plot_metrics.py --sft outputs/sft --dpo outputs/dpo_hh outputs/dpo_uf --lora outputs/lora_persona --output_dir outputs/plots""")
sh(f"""python scripts/collect_results.py --runs_root outputs --output outputs/report_tables.md""")
sh(f"""python scripts/show_examples.py --pairs outputs/eval/dpo_uf_vs_sft_pairs.jsonl --judged outputs/eval/dpo_uf_vs_sft_judged.jsonl --k 3 --output outputs/eval/examples_dpo_uf.md > /dev/null""")
sh(f"""python scripts/show_examples.py --pairs outputs/eval/dpo_hh_vs_sft_pairs.jsonl --judged outputs/eval/dpo_hh_vs_sft_judged.jsonl --k 3 --output outputs/eval/examples_dpo_hh.md > /dev/null""")
sh(f"""python scripts/show_examples.py --pairs outputs/eval/lora_vs_base_persona_pairs.jsonl --judged outputs/eval/lora_vs_base_persona_persona_style_judged.jsonl --k 2 --output outputs/eval/examples_lora_style.md > /dev/null""")

In [ ]:
sh(f"""python scripts/analyze_generations.py outputs/generations/*.json --output outputs/eval/generation_stats.json""")

In [ ]:
sh(f"""cd {REPO_DIR} && zip -qr /kaggle/working/06_final_results.zip outputs && ls -lh /kaggle/working/06_final_results.zip""")